# ICF Notes: Exploration, Assessment Conclusions, and Cohort Linkage

This notebook prepares the physiotherapy and ICF classifier data for the recovery analysis. It loads the clinical notes, describes them, extracts the preoperative risk conclusion that the physiotherapist records, attaches that conclusion to every note as a value that can change over time, and links the result to the oesophagus surgical cohort.

## Goal

For each patient we want two things alongside their surgical outcome: the physiotherapist's preoperative risk conclusion, low or high, and the ICF classifier scores that describe how the patient is functioning across notes over time. Together these let us test whether the assessment and the ICF signal predict complications and recovery.

## Key design decisions

* MDN is the patient key. It is padded to seven digits everywhere, because the leading zero is sometimes dropped administratively, and the surgical cohort key upn is the same number.
* The conclusion is read from the notes themselves. A note that records a risk conclusion sets the conclusion from its date onward, the earliest conclusion also applies to everything before it, and a later conclusion replaces it from its own date. A patient can therefore carry one conclusion before surgery and another after, or an updated one from a repeated assessment.
* Where a patient has no conclusion in their notes, Edwin's manual label is used instead.
* No note is dropped. The full set of notes is kept so the ICF scores remain available per note for the trend analysis.
* The conclusion vocabulary, laag, verhoogd, hoog and so on, is collapsed to a single low versus high scheme that matches Edwin's manual labels, with the granular level kept alongside.
* Saved files are written without the free text column, so the patient and note tables write quickly.

## 1. Setup

In [ ]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, re, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

# Edwin's source data stays in its original location; everything we produce goes into the Thesis folder.
ICF_DIR = Path(r"Z:\Users\predicting recovery\ICF_output")
MANUAL  = Path(r"Z:\Users\predicting recovery\For Edwin\ICF_patients_without_valid_assessment_in_DUCA_or_DLCA_ege.csv")
COHORT  = Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis\data_derived\cohort_clean.csv")
OUT     = Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis\outputs\icf")
OUT.mkdir(parents=True, exist_ok=True)

# The conclusion vocabulary collapsed to one scheme. Per Edwin (15 June 2026): matig (moderate) is low, but matig verhoogd and matig hoog count as high.
RISK_TO_BINARY = {'laag':'low','matig':'low','matig verhoogd':'high','matig hoog':'high',
                  'licht verhoogd':'high','iets verhoogd':'high',
                  'verhoogd':'high','sterk verhoogd':'high','hoog':'high'}
MANUAL_TO_BINARY = {'low':'low','high':'high','none':None}
MANUAL_EXCLUDE_REMARKS = {'trial','only preparation','no surgery','colleague','2 surgeries / assessments'}

def pad7(s):
    return pd.Series(s).astype(str).str.strip().str.replace(r'\.0$','',regex=True).str.zfill(7)
print('Setup complete. Outputs will be written to', OUT)

## 2. Load and combine the note files

In [ ]:
files=sorted(ICF_DIR.glob("ds_project_selection_*_output.csv"))
print(f"Files found: {len(files)}\n")
dfs=[]
for f in files:
    d=pd.read_csv(f, sep=None, engine='python', dtype={'MDN':str}); d['source_file']=f.name; dfs.append(d)
    print(f"  {f.name}: {d.shape}")
notes=pd.concat(dfs, ignore_index=True, sort=False)
print(f"\nCombined: {notes.shape}")

## 3. Normalise the MDN identifier to seven digits

In [ ]:
print("MDN length before normalisation:")
print(notes['MDN'].dropna().astype(str).str.strip().str.len().value_counts().sort_index().to_string())
notes['MDN']=pad7(notes['MDN'])
print("\nMDN length after normalisation:")
print(notes['MDN'].str.len().value_counts().sort_index().to_string())
print(f"\nUnique patients: {notes['MDN'].nunique():,}")

## 4. The data at a glance

Each row is a clinical note. The columns we use are the patient key MDN, the note date, the care provider name and role, the free text, and the ICF classifier scores. The ICF columns end in `_lvl` and hold an ordinal level for a functioning domain.

In [ ]:
notes['NOTITIE_DATUM_dt']=pd.to_datetime(notes['NOTITIE_DATUM'], errors='coerce')
icf_cols=[c for c in notes.columns if c.endswith('_lvl')]
ICF_REFERENCE={'ENR-B1300_lvl':'energy and drive','ATT-B140_lvl':'attention','STM-B152_lvl':'mood',
  'ADM-B440_lvl':'respiration','INS-B455_lvl':'exercise tolerance','MBW-B530_lvl':'weight maintenance',
  'FAC-D540_lvl':'dressing','ETN-D550_lvl':'eating','BER-D840-D859_lvl':'work and employment',
  'SOP-B280_lvl':'pain','SLP-B134_lvl':'sleep','FML-D760_lvl':'family relationships',
  'HLC-B164_lvl':'higher level cognition','MAE-D465_lvl':'moving around'}
ref=pd.DataFrame([{'column':c,'domain':ICF_REFERENCE.get(c,''),'non_null':int(notes[c].notna().sum()),
                   'pct':round(100*notes[c].notna().mean(),1)} for c in icf_cols]).sort_values('pct',ascending=False)
print(f"ICF classifier columns: {len(icf_cols)}\n"); print(ref.to_string(index=False))

### 4.1 Unique patients and notes per patient

In [ ]:
npp=notes.groupby('MDN').size()
print(f"Unique patients: {notes['MDN'].nunique():,}")
print(f"Total notes:     {len(notes):,}")
print(f"Notes per patient: median {npp.median():.0f}, mean {npp.mean():.1f}, p90 {npp.quantile(.9):.0f}, max {npp.max():.0f}")
fig,ax=plt.subplots(figsize=(10,4)); npp.plot(kind='hist',bins=50,color='steelblue',edgecolor='white',ax=ax)
ax.axvline(npp.median(),color='black',label=f'median {npp.median():.0f}'); ax.set_xlabel('notes per patient'); ax.legend()
ax.set_title('Notes per patient'); plt.tight_layout(); plt.savefig(OUT/'notes_per_patient.png',dpi=110); plt.close(fig)

### 4.2 Care provider roles and temporal coverage

In [ ]:
print("Top care provider roles (ZORGV_TYPE):")
print(notes['ZORGV_TYPE'].value_counts(dropna=False).head(8).to_string())
yr=notes['NOTITIE_DATUM_dt'].dt.year.value_counts().sort_index()
print("\nNotes per year:"); print(yr.to_string())
fig,ax=plt.subplots(figsize=(10,4)); yr.plot(kind='bar',color='steelblue',edgecolor='white',ax=ax)
ax.set_title('Notes per year'); ax.set_xlabel('year'); plt.tight_layout(); plt.savefig(OUT/'notes_per_year.png',dpi=110); plt.close(fig)

## 5. Extract the Conclusie from each note

The physiotherapy assessment ends with a Conclusie that states the preoperative risk. We take the last Conclusie in a note, the text running from the word Conclusie up to the word plan, capped at 1000 characters when plan is absent. This mirrors how the form is written, conclusion then plan.

In [ ]:
CONCLUSIE_PATTERN=re.compile(r'\bconclusie\b\s*[:\-\.\s]*(.*?)(?=\bplan\b|\Z)', flags=re.IGNORECASE|re.DOTALL)
MAX_CONCLUSIE_CHARS=1000
def extract_conclusie(tekst):
    if not isinstance(tekst,str) or not tekst.strip(): return None, False
    m=CONCLUSIE_PATTERN.findall(tekst)
    if not m: return None, False
    body=m[-1].strip()
    if len(body)<5: return None, False
    if len(body)>MAX_CONCLUSIE_CHARS: return body[:MAX_CONCLUSIE_CHARS].strip(), True
    return body, False
_res=notes['TEKST'].apply(extract_conclusie)
notes['CONCLUSIE']=_res.str[0]; notes['CONCLUSIE_truncated']=_res.str[1]
print(f"Notes with a Conclusie: {notes['CONCLUSIE'].notna().sum():,} / {len(notes):,} "
      f"({notes['CONCLUSIE'].notna().mean()*100:.1f}%)")
print(f"  of those, hit the {MAX_CONCLUSIE_CHARS}-char cap (no plan anchor): {int(notes['CONCLUSIE_truncated'].sum()):,}")
print(f"Patients with at least one Conclusie: {notes.loc[notes['CONCLUSIE'].notna(),'MDN'].nunique():,}")

## 6. Classify the risk level and identify assessment notes

The risk level is read from the Conclusie by matching a risk word, laag, matig, verhoogd and its variants, or hoog, followed by a risk noun. A note counts as a preoperative assessment when it carries the form markers, or a risk phrase, or a fitness or trainability phrase, and is not explicitly postoperative, and has a Conclusie. The conclusion we propagate is taken from the physiotherapy assessment notes. The risk word matching tolerates the common typos seen in the notes.

In [ ]:
RISK_KEYWORDS=[r'laag',r'matig\s+verhoogd',r'matig\s+hoog',r'matig',r'licht\s+verhoogd',r'iets\s+verhoogd',r'sterk\s+verhoogd',r'verhoogd',r'hoog']
RISK_NOUN=r'(?:risico|risiko|risicio|riisico|riisco|kans)'
def classify_risk(conclusie):
    if not isinstance(conclusie,str): return 'unspecified'
    t=conclusie.lower()
    for kw in RISK_KEYWORDS:
        if re.search(r'\b'+kw+r'\b\s*(?:e?\s+)?'+RISK_NOUN,t): return re.sub(r'\s+',' ',kw).replace('\\s+',' ')
    return 'unspecified'
notes['risk_level']=notes['CONCLUSIE'].apply(classify_risk)

ASSESSMENT_MARKERS=r'\b(?:risicofactor\s+indien|indicatie|pre[\-\s]operatief)\b'
RISK_CLASSIFICATION=(r'\b(?:laag|matig|hoog|verhoogd|licht\s+verhoogd|iets\s+verhoogd|sterk\s+verhoogd)\s*(?:e?\s+)?'+RISK_NOUN+r'\b')
FITNESS_TRAINABILITY=(r'\b(?:(?:goede?|matig(?:e)?|redelijk(?:e)?|slecht(?:e)?|lage?)\s+fitheid'
                      r'|trainba[ar][r]?'
                      r'|(?:goede?|matig(?:e)?|redelijk(?:e)?|slecht(?:e)?|lage?)\s+belastbaarheid'
                      r'|fysieke?\s+fitheid)\b')
POSTOP_EXCLUSION=(r'\b(?:heden\s+post[-\s]?operatief|ondanks\s+recente\s+OK|mooi\s+herstel|maakt\s+al\s+stapjes)\b')
def is_assessment(df):
    has_form_marker=df['TEKST'].astype(str).str.contains(ASSESSMENT_MARKERS,regex=True,case=False,na=False)
    has_risk_phrase=df['CONCLUSIE'].astype(str).str.contains(RISK_CLASSIFICATION,regex=True,case=False,na=False)
    has_fitness_phrase=df['CONCLUSIE'].astype(str).str.contains(FITNESS_TRAINABILITY,regex=True,case=False,na=False)
    is_postop=df['CONCLUSIE'].astype(str).str.contains(POSTOP_EXCLUSION,regex=True,case=False,na=False)
    return (has_form_marker | has_risk_phrase | has_fitness_phrase) & ~is_postop & df['CONCLUSIE'].notna()
notes['is_assessment']=is_assessment(notes)
notes['is_physio']=notes['ZORGV_SPECCODE']==400.0
edwin_variants=['GELEIJN, E.','GELEIJN,E.','GELEIJN, E','GELEIJN E.']
notes['is_edwin']=notes['ZORGV_NAAM'].astype(str).str.upper().str.strip().isin([v.upper() for v in edwin_variants])

# Three assessment scopes, as in the established exploration
all_a=notes['is_assessment']; phys_a=notes['is_assessment'] & notes['is_physio']; edw_a=notes['is_assessment'] & notes['is_edwin']
print(f"Assessment notes, any provider:   {int(all_a.sum()):,} over {notes.loc[all_a,'MDN'].nunique():,} patients")
print(f"Assessment notes, physiotherapy:  {int(phys_a.sum()):,} over {notes.loc[phys_a,'MDN'].nunique():,} patients")
print(f"Assessment notes, Edwin:          {int(edw_a.sum()):,} over {notes.loc[edw_a,'MDN'].nunique():,} patients")
print("\nRisk level distribution on physiotherapy assessment notes:")
print(notes.loc[phys_a,'risk_level'].value_counts().to_string())
fig,ax=plt.subplots(figsize=(9,4)); notes.loc[phys_a,'risk_level'].value_counts().plot(kind='bar',color='steelblue',edgecolor='white',ax=ax)
ax.set_title('Risk level distribution (physiotherapy assessment notes)'); plt.tight_layout(); plt.savefig(OUT/'risk_level_distribution.png',dpi=110); plt.close(fig)

## 7. The conclusion as a value that changes over time

A note that carries a risk classification is a conclusion point. Each note then inherits the conclusion that was current at its date: the most recent conclusion on or before the note. The earliest conclusion also applies to every note before it, and when a later conclusion appears it takes over from its own date. This is why a patient can read low before surgery and high afterwards, or carry an updated conclusion from a repeated assessment.

In [ ]:
events=notes.loc[notes['is_physio'] & notes['is_assessment'] & (notes['risk_level']!='unspecified')
                 & notes['NOTITIE_DATUM_dt'].notna(),
                 ['MDN','NOTITIE_DATUM_dt','risk_level']].copy()
events['risk_binary']=events['risk_level'].map(RISK_TO_BINARY)
events=events.dropna(subset=['risk_binary']).drop_duplicates(['MDN','NOTITIE_DATUM_dt']).sort_values('NOTITIE_DATUM_dt')
evpp=events.groupby('MDN').size()
print(f"Patients with a conclusion in their notes: {events['MDN'].nunique():,}")
print(f"  one conclusion: {(evpp==1).sum():,} | two: {(evpp==2).sum():,} | three or more: {(evpp>=3).sum():,}")

notes_dated=notes.dropna(subset=['NOTITIE_DATUM_dt']).sort_values('NOTITIE_DATUM_dt').copy()
ev=events[['MDN','NOTITIE_DATUM_dt','risk_binary','risk_level']].rename(columns={'risk_binary':'asof_bin','risk_level':'asof_lvl'})
merged=pd.merge_asof(notes_dated, ev, on='NOTITIE_DATUM_dt', by='MDN', direction='backward')
merged['conc_binary']=merged.groupby('MDN')['asof_bin'].bfill()
merged['conc_level']=merged.groupby('MDN')['asof_lvl'].bfill()
merged['conc_source']=np.where(merged['conc_binary'].notna(),'note',None)
print(f"\nNotes with a conclusion from the notes: {merged['conc_binary'].notna().sum():,}")

## 8. Fall back to Edwin's manual label where the notes carry no conclusion

In [ ]:
man=pd.read_csv(MANUAL, sep=';'); man.columns=[c.strip() for c in man.columns]
lab_col=[c for c in man.columns if 'high' in c.lower() and 'low' in c.lower()][0]
man['MDN']=pad7(man['MDN'])
man['remark_clean']=man['remark'].astype(str).str.strip().str.lower().replace('nan','')
man=man[~man['remark_clean'].isin({r.lower() for r in MANUAL_EXCLUDE_REMARKS})]
man['manual_binary']=man[lab_col].astype(str).str.strip().str.lower().map(MANUAL_TO_BINARY)
manual_per_mdn=man.dropna(subset=['manual_binary']).drop_duplicates('MDN')[['MDN','manual_binary']]
print(f"Patients with a usable manual label: {manual_per_mdn['MDN'].nunique():,}")

merged=merged.merge(manual_per_mdn, on='MDN', how='left')
fill=merged['conc_binary'].isna() & merged['manual_binary'].notna()
merged.loc[fill,'conc_binary']=merged.loc[fill,'manual_binary']
merged.loc[fill,'conc_level']=merged.loc[fill,'manual_binary']
merged.loc[fill,'conc_source']='manual'
merged['is_last_note']=False
merged.loc[merged.groupby('MDN')['NOTITIE_DATUM_dt'].tail(1).index,'is_last_note']=True
print(f"\nPatients with a conclusion after the fallback: {merged.loc[merged['conc_binary'].notna(),'MDN'].nunique():,}")
print("\nNote level conclusion:"); print(merged['conc_binary'].value_counts(dropna=False).to_string())
print("\nBy source:"); print(merged['conc_source'].value_counts(dropna=False).to_string())

## 9. Per patient summary and ICF coverage

In [ ]:
def summarise(g):
    gs=g.sort_values('NOTITIE_DATUM_dt')
    return pd.Series({'n_notes':len(g),'n_conclusions':int(g['asof_bin'].notna().sum()),
        'final_conclusion':gs['conc_binary'].dropna().iloc[-1] if gs['conc_binary'].notna().any() else None,
        'source':g['conc_source'].dropna().iloc[0] if g['conc_source'].notna().any() else None,
        'changes_over_time':g['conc_binary'].dropna().nunique()>1})
patient_summary=merged.groupby('MDN').apply(summarise).reset_index()
print(f"Patients whose conclusion changes over time: {int(patient_summary['changes_over_time'].sum()):,}")
print("\nFinal conclusion per patient:"); print(patient_summary['final_conclusion'].value_counts(dropna=False).to_string())
cov=pd.DataFrame([{'icf_dim':c,'domain':ICF_REFERENCE.get(c,''),
                   'patients_with_any':int((merged.groupby('MDN')[c].apply(lambda s:s.notna().sum())>0).sum())}
                  for c in icf_cols]).sort_values('patients_with_any',ascending=False)
print("\nICF dimension coverage:"); print(cov.to_string(index=False))

## 10. Link to the oesophagus surgical cohort

In [ ]:
coh=pd.read_csv(COHORT); coh['MDN']=pad7(coh['upn'])
icf_mdns=set(merged['MDN']); concl_mdns=set(merged.loc[merged['conc_binary'].notna(),'MDN']); oeso=set(coh['MDN'])
print(f"Oesophagus patients:        {len(oeso):,}")
print(f"  with ICF notes:           {len(oeso & icf_mdns):,}")
print(f"  with a conclusion:        {len(oeso & concl_mdns):,}")
linked=coh[coh['MDN'].isin(concl_mdns)].merge(patient_summary[['MDN','final_conclusion','source','changes_over_time']],on='MDN',how='left')
print("\nConclusion among linked oesophagus patients:"); print(linked['final_conclusion'].value_counts(dropna=False).to_string())

## 11. Save the prepared datasets into the Thesis folder

In [ ]:
slim_cols=['MDN','NOTITIE_DATUM_dt','source_file','is_assessment','is_edwin','is_last_note',
           'conc_binary','conc_level','conc_source']+icf_cols
merged[slim_cols].to_csv(OUT/'icf_notes_with_conclusion.csv', index=False, encoding='utf-8')
patient_summary.to_csv(OUT/'conclusion_per_patient.csv', index=False, encoding='utf-8')
cov.to_csv(OUT/'icf_coverage.csv', index=False, encoding='utf-8')
linked.to_csv(OUT/'oesophagus_with_conclusion.csv', index=False, encoding='utf-8')
print("Saved to", OUT)
for f in ['icf_notes_with_conclusion.csv','conclusion_per_patient.csv','icf_coverage.csv','oesophagus_with_conclusion.csv']:
    print('  ', f)
print("\nThe note level file holds the ICF scores and the conclusion without the free text, so it writes quickly.")

## 12. What this prepares for

The note level file keeps every note with its ICF scores and the conclusion that applied at that date, the per patient file gives one row per patient with the final conclusion and whether it changed, and the linked file joins the conclusion to the oesophagus outcomes.

The next notebook brings in the surgery date from the surgical cohort to mark the preoperative notes, builds the preoperative ICF trend features per domain, and runs the analysis: whether the conclusion predicts pulmonary complications, whether the ICF scores predict complications and add to the surgical model, and how the conclusion and the ICF signal compare with Edwin's manual labels.